# HairFastGAN vs Barbershop-FT — Common evaluation
**Colab ใหม่ → GPU → Run all → เชื่อม Drive**

ประเมิน baseline และ A–D **epoch30 ของทั้งสองโมเดล** ด้วยฟังก์ชันเดียวกันบน validation9 และ test_candidate10 ไม่มีการฝึกหรือเลือกผู้ชนะจาก test

การกำหนด epoch30 นี้ทำหลังเห็นผลเดิม จึงเป็น **retrospective fixed-epoch comparison** ไม่ใช่ best checkpoint ด้วยเกณฑ์ใหม่ และ test เดิมเคยถูกดูแล้ว ไม่ใช่ชุดยืนยันใหม่

เงื่อนไขฝึกย้อนหลังยังต่างกัน: โมดูล, loss บางส่วน และ detail gate ต่างกัน สมุดนี้แก้ส่วนประเมิน ไม่เปลี่ยนประวัติฝึก งานวิจัยต้องเรียกโมเดลที่ดัดแปลงว่า **Barbershop-FT**

เมทริกหลัก: L/ab, ArcFace cosine, protected MAE/SSIM/PSNR ใช้ฟังก์ชันและ input/mask เดียวกัน เมทริกทรง IoU/Dice/Boundary F1 แสดงแยกเป็น **HFG-derived proxy** ไม่ใช่ ground truth อิสระ

ไม่รวม loss, detail-to-own-baseline หรือ normalized score คนละฐานในตารางจัดอันดับข้ามโมเดล

ต้องมี checkpoint epoch30 + latent cache เดิมของ Barbershop ทั้ง validation และ test หากขาดจะหยุด ไม่แทนด้วย best.pt ไม่ฝึกหรือสร้าง Barbershop latent ใหม่เงียบ ๆ


In [ ]:
from google.colab import drive, files
from pathlib import Path
import os,sys,json,hashlib,shutil,subprocess,uuid,time,copy,gc,random,zipfile

drive.mount('/content/drive')
FT_ROOT=Path('/content/drive/MyDrive/Barbershop_FT_30Epoch/27f7f146ef0d')
STUDY_ROOT=Path('/content/drive/MyDrive/HairFastGAN_BPP_4Trials_PretrainedStart/9be83b9c2e')
BARBER_WEIGHTS=Path('/content/drive/MyDrive/Barbershop_matched_weights')
SOURCE_OVERRIDE=''
DATASET_ZIP='' # ว่าง: อัปโหลด HairFastGAN_Dataset_Split_v2.zip เมื่อถึงขั้นเตรียม test
RUN_TEST=True
OUTPUT_ROOT=Path('/content/drive/MyDrive/Common_HFG_Barbershop_Evaluation')

def sha(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for chunk in iter(lambda:f.read(8*1024**2),b''):h.update(chunk)
 return h.hexdigest()
def need(p):
 if not p.is_file():raise FileNotFoundError('ต้องใช้ไฟล์เดิม: '+str(p))
 return p
FT_PROTOCOL=json.loads(need(FT_ROOT/'protocol.json').read_text());FT_ID=FT_ROOT.name
MANIFESTS={t:json.loads(need(STUDY_ROOT/f'trial_{t}/manifest.json').read_text()) for t in 'ABCD'}
REFERENCE_META=MANIFESTS['A'];SOURCE=Path(SOURCE_OVERRIDE or REFERENCE_META['source'])
VAL_KEYS=FT_PROTOCOL['validation_keys'];assert len(VAL_KEYS)==9
assert sha(need(SOURCE/'manifest.json'))==FT_PROTOCOL['source_manifest_sha256']
assert sha(need(BARBER_WEIGHTS/'ffhq.pt'))==FT_PROTOCOL['ffhq_sha256']
BARBER_ROOT=FT_ROOT/'front_end';CROPS=BARBER_ROOT/'shared_crops';FT_TRIALS=FT_PROTOCOL['trials']
for t,m in MANIFESTS.items():
 assert m['config']['train_keys']==FT_PROTOCOL['train_keys'] and m['config']['validation_keys']==VAL_KEYS
 assert m['config']['epochs']==30 and m['config']['batch_size']==1
for k in VAL_KEYS:
 assert sha(need(SOURCE/'cache'/(k+'.pt')))==FT_PROTOCOL['source_cache_hashes'][k]
 for mask,digest in FT_PROTOCOL['mask_hashes'][k].items():assert sha(need(STUDY_ROOT/'trial_A/inputs'/k/(mask+'.png')))==digest
BS_CASES={};FRONT_INDEX={}
for split in ['validation']+(['test_candidate'] if RUN_TEST else []):
 cfg=json.loads(need(BARBER_ROOT/split/'worker_config.json').read_text());assert cfg['protocol_id']==FT_ID
 BS_CASES[split]=cfg['cases']
 records=json.loads(need(BARBER_ROOT/split/'results.json').read_text())
 FRONT_INDEX[split]={r['case']:r for r in records}
 for row in BS_CASES[split]:
  r=FRONT_INDEX[split][row['key']];need(Path(r['final_float']))
  assert len(list((Path(r['final_png']).parent/'Blend_realistic').glob('*.npz')))==1
  for role in ['face','shape','color']:need(CROPS/(row[role]+'.png'))
OUTPUT_ROOT.mkdir(exist_ok=True,parents=True)
OUT=OUTPUT_ROOT/('common_'+time.strftime('%Y%m%d_%H%M%S')+'_'+uuid.uuid4().hex[:6]);OUT.mkdir()
print('Output:',OUT)


## ติดตั้งและใช้ตัวประเมินชุดเดียวกัน

In [ ]:
%pip install -q opencv-python matplotlib scipy scikit-image scikit-learn pandas tqdm dlib face-alignment gdown addict wandb torchmetrics dill ninja transformers diffusers accelerate fastapi uvicorn python-multipart pyngrok nest-asyncio huggingface_hub
%pip install -q git+https://github.com/openai/CLIP.git
import torch
assert torch.cuda.is_available(), 'เลือก Runtime > Change runtime type > GPU ก่อนรัน'
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)


In [ ]:
from google.colab import drive
from pathlib import Path
import os, shutil, subprocess, sys, json, time
from huggingface_hub import snapshot_download


LOCAL_REPO = '/content/HairFastGAN_common_eval'
DRIVE_CACHE_DIR = Path('/content/drive/MyDrive/HairFastGAN_cache/pretrained_models')

# ใส่ commit ที่บันทึกใน metadata.json เมื่อต้องการทำซ้ำภายหลัง
HAIRFAST_REVISION = '49e98019c9243d1991c19d45764fc3340a498de2'
if not Path(LOCAL_REPO, '.git').exists():
    subprocess.run(['git', 'clone', 'https://github.com/AIRI-Institute/HairFastGAN.git', LOCAL_REPO], check=True)
if HAIRFAST_REVISION:
    subprocess.run(['git', '-C', LOCAL_REPO, 'checkout', HAIRFAST_REVISION], check=True)
os.chdir(LOCAL_REPO)
local_models = Path(LOCAL_REPO, 'pretrained_models')
if not local_models.exists() or not any(local_models.iterdir()):
    if DRIVE_CACHE_DIR.exists() and any(DRIVE_CACHE_DIR.iterdir()):
        shutil.copytree(DRIVE_CACHE_DIR, local_models, dirs_exist_ok=True)
    else:
        snapshot = snapshot_download('AIRI-Institute/HairFastGAN', allow_patterns=['pretrained_models/**'])
        shutil.copytree(Path(snapshot, 'pretrained_models'), local_models, dirs_exist_ok=True)

# ตรวจ cache สำคัญ รวมถึงกรณีได้ Git-LFS pointer แทน weights
for relative in ['StyleGAN/ffhq.pt', 'encoder4editing/e4e_ffhq_encode.pt']:
    p = local_models / relative
    if not p.exists() or p.stat().st_size < 1000000:
        raise RuntimeError(f'โมเดลขาดหรือไม่สมบูรณ์: {p} โปรดตรวจ cache ก่อนดำเนินการ')
Path('input').mkdir(exist_ok=True)
Path('output').mkdir(exist_ok=True)
REPO_COMMIT = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print('Repo commit:', REPO_COMMIT)


In [ ]:
# Compatibility backend: native torch operators also execute on CUDA tensors.
# No PyTorch/CUDA version downgrade; no changes to weights or transfer parameters.
import ast, hashlib, importlib, sys
from pathlib import Path
STYLEGAN_OPS_BACKEND = 'torch_native'

def _native_stylegan_source(source, kind):
    marker = '# HAIRFAST_TORCH_NATIVE_V1'
    if marker in source:
        return source
    tree = ast.parse(source)
    extension = 'fused' if kind == 'fused_act' else 'upfirdn2d_op'
    entry = 'fused_leaky_relu' if kind == 'fused_act' else 'upfirdn2d'
    edits = []
    found_load = found_entry = False
    for node in tree.body:
        if isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == extension for t in node.targets):
            if not isinstance(node.value, ast.Call) or not isinstance(node.value.func, ast.Name) or node.value.func.id != 'load':
                raise RuntimeError('รูปแบบ extension เปลี่ยนไป: ' + kind)
            edits.append((node.lineno-1, node.end_lineno, extension + ' = None\n'))
            found_load = True
        if isinstance(node, ast.FunctionDef) and node.name == entry:
            edits.append((node.lineno-1, node.end_lineno, FUSED_NATIVE if kind == 'fused_act' else UP_NATIVE))
            found_entry = True
    if not found_load or not found_entry:
        raise RuntimeError('ไม่พบจุด patch ที่คาดไว้ใน ' + kind)
    lines = source.splitlines(True)
    for start,end,text in sorted(edits, reverse=True):
        lines[start:end] = [text]
    updated = marker + '\n' + ''.join(lines)
    if kind == 'upfirdn2d' and not any(isinstance(n, ast.FunctionDef) and n.name == 'upfirdn2d_native' for n in tree.body):
        updated += '\n\n' + UP_NATIVE_IMPL + '\n'
    compile(updated, kind + '.py', 'exec')
    return updated

FUSED_NATIVE = 'def fused_leaky_relu(input, bias=None, negative_slope=0.2, scale=2 ** 0.5):\n    if bias is not None:\n        input = input + bias.reshape(1, -1, *([1] * (input.ndim - 2)))\n    return torch.nn.functional.leaky_relu(input, negative_slope=negative_slope) * scale\n'
UP_NATIVE = 'def upfirdn2d(input, kernel, up=1, down=1, pad=(0, 0)):\n    return upfirdn2d_native(input, kernel, up, up, down, down,\n                           pad[0], pad[1], pad[0], pad[1])\n'
UP_NATIVE_IMPL = 'def upfirdn2d_native(\n    input, kernel, up_x, up_y, down_x, down_y, pad_x0, pad_x1, pad_y0, pad_y1\n):\n    _, channel, in_h, in_w = input.shape\n    input = input.reshape(-1, in_h, in_w, 1)\n\n    _, in_h, in_w, minor = input.shape\n    kernel_h, kernel_w = kernel.shape\n\n    out = input.view(-1, in_h, 1, in_w, 1, minor)\n    out = F.pad(out, [0, 0, 0, up_x - 1, 0, 0, 0, up_y - 1])\n    out = out.view(-1, in_h * up_y, in_w * up_x, minor)\n\n    out = F.pad(\n        out, [0, 0, max(pad_x0, 0), max(pad_x1, 0), max(pad_y0, 0), max(pad_y1, 0)]\n    )\n    out = out[\n        :,\n        max(-pad_y0, 0) : out.shape[1] - max(-pad_y1, 0),\n        max(-pad_x0, 0) : out.shape[2] - max(-pad_x1, 0),\n        :,\n    ]\n\n    out = out.permute(0, 3, 1, 2)\n    out = out.reshape(\n        [-1, 1, in_h * up_y + pad_y0 + pad_y1, in_w * up_x + pad_x0 + pad_x1]\n    )\n    w = torch.flip(kernel, [0, 1]).view(1, 1, kernel_h, kernel_w)\n    out = F.conv2d(out, w)\n    out = out.reshape(\n        -1,\n        minor,\n        in_h * up_y + pad_y0 + pad_y1 - kernel_h + 1,\n        in_w * up_x + pad_x0 + pad_x1 - kernel_w + 1,\n    )\n    out = out.permute(0, 2, 3, 1)\n    out = out[:, ::down_y, ::down_x, :]\n\n    out_h = (in_h * up_y + pad_y0 + pad_y1 - kernel_h) // down_y + 1\n    out_w = (in_w * up_x + pad_x0 + pad_x1 - kernel_w) // down_x + 1\n\n    return out.view(-1, channel, out_h, out_w)'

def install_native_stylegan_ops(repo):
    root = Path(repo).resolve()
    pairs = sorted((root / 'models').rglob('op/fused_act.py'))
    if not pairs:
        raise RuntimeError('ไม่พบ StyleGAN operator ใน ' + str(root))
    planned = []
    for fused_path in pairs:
        for path,kind in [(fused_path,'fused_act'), (fused_path.with_name('upfirdn2d.py'),'upfirdn2d')]:
            if not path.is_file():
                raise RuntimeError('ขาด operator: ' + str(path))
            module_name = '.'.join(path.relative_to(root).with_suffix('').parts)
            if module_name in sys.modules:
                raise RuntimeError('StyleGAN ถูก import แล้ว: Restart runtime แล้วรันตามลำดับใหม่')
            old = path.read_text()
            planned.append((path, old, _native_stylegan_source(old,kind)))
    # Validate every candidate before writing; preserve each original once.
    records = []
    for path,old,new in planned:
        if old != new:
            backup = path.with_suffix('.py.before_native')
            if not backup.exists(): backup.write_text(old)
            path.write_text(new)
        records.append({'path':str(path.relative_to(root)),
                        'sha256':hashlib.sha256(new.encode()).hexdigest()})
    importlib.invalidate_caches()
    return records

STYLEGAN_NATIVE_PATCHES = install_native_stylegan_ops(LOCAL_REPO)
print('StyleGAN backend:', STYLEGAN_OPS_BACKEND, '| patched files:', len(STYLEGAN_NATIVE_PATCHES))



In [ ]:
from PIL import Image
if not hasattr(Image,"ANTIALIAS"): Image.ANTIALIAS=Image.Resampling.LANCZOS
import sys, torch
sys.path.insert(0, LOCAL_REPO)
from hair_swap import HairFast, get_parser
hair_fast = HairFast(get_parser().parse_args([]))
print('HairFastGAN พร้อม')

import numpy as np,pandas as pd
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from models.Net import get_segmentation as segment
from IPython.display import display
b=hair_fast.blend;enc=b.blending_encoder;G=b.net.generator;pp=b.post_process;align=hair_fast.align
for module in [enc,G,pp,align]:module.eval().requires_grad_(False)
assert sha(b.opts.blending_checkpoint)==REFERENCE_META['init_blending_sha256']
assert sha(b.opts.pp_checkpoint)==REFERENCE_META['init_postprocess_sha256']


In [ ]:
runtime_baseS={}
def cpu(x):
    if torch.is_tensor(x):
        return x.detach().cpu().clone()
    if isinstance(x, dict):
        return {k: cpu(v) for k, v in x.items()}
    if isinstance(x, list):
        return [cpu(v) for v in x]
    if isinstance(x, tuple):
        return tuple((cpu(v) for v in x))
    return copy.deepcopy(x)

def gpu(x):
    if torch.is_tensor(x):
        return x.to('cuda').clone()
    if isinstance(x, dict):
        return {k: gpu(v) for k, v in x.items()}
    if isinstance(x, list):return [gpu(v) for v in x]
    if isinstance(x, tuple):return tuple(gpu(v) for v in x)
    return x

def resize(x):
    return F.interpolate(x, size=(256, 256), mode='bilinear', align_corners=False)

def saveim(x, p):
    a = x.detach().float().cpu()[0].clamp(0, 1).permute(1, 2, 0).numpy()
    Image.fromarray((a * 255).round().astype('uint8')).save(p)

def lab(x):
    x = x.float().clamp(0, 1)
    lin = torch.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055).pow(2.4))
    M = x.new_tensor([[0.4124564, 0.3575761, 0.1804375], [0.2126729, 0.7151522, 0.072175], [0.0193339, 0.119192, 0.9503041]])
    xyz = torch.einsum('ij,bjhw->bihw', M, lin) / x.new_tensor([0.95047, 1.0, 1.08883])[None, :, None, None]
    f = torch.where(xyz > 0.008856, xyz.clamp_min(1e-08).pow(1 / 3), 7.787 * xyz + 16 / 116)
    return torch.stack([116 * f[:, 1] - 16, 500 * (f[:, 0] - f[:, 1]), 200 * (f[:, 1] - f[:, 2])], 1)

def render(s, d):
    S = torch.cat([d['S'][:, :6], s], 1)
    return G([S], input_is_latent=True, return_latents=False, start_layer=4, end_layer=8, layer_in=d['F'], randomize_noise=False)[0]

def finish(raw, d):
    sf, ff = pp(d['I1'], b.downsample_256(raw))
    return G([sf], input_is_latent=True, return_latents=False, start_layer=5, end_layer=8, layer_in=ff, randomize_noise=False)[0]

def detail(x):
    y = x.mean(1, keepdim=True)
    mu = F.avg_pool2d(y, 9, 1, 4)
    v = F.avg_pool2d((y - mu).square(), 9, 1, 4)
    return (y - mu) / (v + 0.001).sqrt()

def unit(raw):return resize(((raw+1)/2).clamp(0,1))
def read_mask(p):
    with Image.open(p) as im:
        if im.size!=(256,256):raise ValueError('Mask size mismatch')
        a=np.asarray(im.convert('L'))>127
    return torch.from_numpy(a.copy())[None,None]

def load_case(k):
    d=torch.load(SOURCE/'cache'/(k+'.pt'),map_location='cpu',weights_only=True)
    folder=STUDY_ROOT/'trial_A/inputs'/k
    for name in ['eval_core','eval_refmask','target','protected']:
        mask=read_mask(folder/(name+'.png'))
        if name in ['target','protected']:
            assert torch.equal(d[name].bool(),mask), (k,name)
        d[name]=mask
    for name in ['core','refmask']:d[name]=d[name].bool()
    for name in ['eval_core','eval_refmask']:assert d[name].sum()>=100
    return d

def quantile_color(im,ref,mask,refmask):
    l=lab(im)[0,:,mask[0,0].bool()];r=lab(ref)[0,:,refmask[0,0].bool()]
    q=torch.linspace(.1,.9,9,device=im.device)
    diff=(torch.quantile(l,q,dim=1)-torch.quantile(r,q,dim=1)).abs()
    return diff[:,0].mean(),diff[:,1:].mean()

def masked_mae(a,b,m):return ((a-b).abs()*m.float()).sum()/(a.shape[1]*m.sum().clamp_min(1))

def shape_metrics(im,d):
    labels=segment(align.to_bisenet(im))
    pred=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')>.5
    target=d['target']
    return dict(target_mask_IoU=float((pred&target).sum()/(pred|target).sum().clamp_min(1)),
                target_missing_fraction=float((target&~pred).sum()/target.sum().clamp_min(1)))


# Frozen differentiable evaluators: reuse exact parser weights, load original ArcFace.
from models.CtrlHair.external_code.face_parsing.my_parsing_util import FaceParsing, FaceParsing_tensor
from losses.pp_losses import Backbone
from huggingface_hub import hf_hub_download
id_path=Path('pretrained_models/ArcFace/ir_se50.pth')
if not id_path.is_file():
    downloaded=hf_hub_download(repo_id='AIRI-Institute/HairFastGAN',filename=str(id_path))
    id_path.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(downloaded,id_path)
id_net=Backbone(input_size=112,num_layers=50,drop_ratio=.6,mode='ir_se')
id_net.load_state_dict(torch.load(id_path,map_location='cpu',weights_only=True),strict=True)
id_net=id_net.cuda().eval().requires_grad_(False)
FaceParsing_tensor.parsing_img()
parser_net=FaceParsing.bise_net
if parser_net is None:raise RuntimeError('Original BiSeNet is not initialized')
parser_net.eval().requires_grad_(False)
assert FaceParsing.label_list[17]=='hair' # raw parser label=17; remapped target label=13

def id_features(im):
    x=F.interpolate(im,size=(256,256),mode='bilinear',align_corners=False)*2-1
    x=F.adaptive_avg_pool2d(x[:,:,35:223,32:220],(112,112))
    feats=id_net(x)
    return F.normalize(feats,dim=1)

def constraint_terms(im,d,fx):
    # Do NOT call get_segmentation: its no_grad + argmax would sever training gradients.
    logits=checkpoint(lambda x:parser_net(align.to_bisenet(x))[0],im,use_reentrant=False) if torch.is_grad_enabled() else parser_net(align.to_bisenet(im))[0]
    other=torch.cat((logits[:,:17],logits[:,18:]),1)
    binary=logits[:,17:18]-torch.logsumexp(other,dim=1,keepdim=True)
    target=F.interpolate(d['target'].float(),size=binary.shape[-2:],mode='nearest')
    if target.sum()==0 or (1-target).sum()==0:raise RuntimeError('Degenerate shape target')
    prob=binary.sigmoid()
    bce=F.binary_cross_entropy_with_logits(binary,target,reduction='none')
    balanced=.5*((bce*target).sum()/target.sum()+(bce*(1-target)).sum()/(1-target).sum())
    dice=1-(2*(prob*target).sum()+1)/(prob.sum()+target.sum()+1)
    embedding=checkpoint(id_features,im,use_reentrant=False) if torch.is_grad_enabled() else id_features(im)
    identity=(1-(embedding*fx['face_embedding']).sum(1)).mean()
    return identity,balanced+dice,balanced,dice

def to_size(x,size):
    return F.interpolate(x,size=(size,size),mode='bilinear',align_corners=False)

def detail_terms(full,d,fx):
    values={}
    for size in [256,512]:
        x=to_size(full,size)
        teacher=fx['pre'] if size==256 else fx['pre_512']
        mask=F.interpolate(d['eval_core'].float(),size=(size,size),mode='nearest')
        inner=-F.max_pool2d(-mask,9,1,4)
        if inner.sum()<64:raise RuntimeError('Insufficient interior hair pixels at '+str(size))
        values[size]=masked_mae(detail(x),detail(teacher),inner)
    return values


from scipy.ndimage import binary_erosion,binary_dilation
from skimage.metrics import structural_similarity
_basic_shape_metrics=shape_metrics
def shape_metrics(im,d):
    result=_basic_shape_metrics(im,d)
    labels=segment(align.to_bisenet(im));p=F.interpolate((labels==13).float().reshape(1,1,*labels.shape[-2:]),(256,256),mode='nearest')[0,0].cpu().numpy()>.5
    t=d['target'][0,0].cpu().numpy()>.5
    result['target_Dice']=float(2*(p&t).sum()/max(p.sum()+t.sum(),1))
    a=p^binary_erosion(p);bnd=t^binary_erosion(t)
    prec=(a&binary_dilation(bnd,iterations=2)).sum()/max(a.sum(),1)
    rec=(bnd&binary_dilation(a,iterations=2)).sum()/max(bnd.sum(),1)
    result['boundary_F1_2px']=float(2*prec*rec/max(prec+rec,1e-12))
    return result

def extra_metrics(im,d):
    x=im[0].permute(1,2,0).cpu().numpy();y=d['face'][0].permute(1,2,0).cpu().numpy()
    mask=d['protected'][0,0].cpu().numpy()>.5
    _,ss=structural_similarity(x,y,data_range=1.,channel_axis=2,full=True)
    safe=binary_erosion(mask,iterations=3)
    if not safe.any():raise ValueError('No protected SSIM interior')
    mse=float(((x-y)**2)[mask].mean())
    return dict(protected_PSNR=float(-10*np.log10(max(mse,1e-12))),protected_SSIM=float(ss[safe].mean()),
        any_channel_255_fraction=float((im[0,:,d['eval_core'][0,0]].max(0).values>=1-0.5/255).float().mean()))


In [ ]:
def joint_forward(d,grad=False):
    def blend_fn(a,c,x,y):return enc(a,c,x,y)
    args=[d['args'][i] for i in range(4)]
    s=checkpoint(blend_fn,*args,use_reentrant=False) if grad else blend_fn(*args)
    raw_pre=checkpoint(lambda a:render(a,d),s,use_reentrant=False) if grad else render(s,d)
    pp_in=b.downsample_256(raw_pre)
    sf,ff=checkpoint(lambda a,c:pp(a,c),d['I1'],pp_in,use_reentrant=False) if grad else pp(d['I1'],pp_in)
    def generate(a,c):return G([a],input_is_latent=True,return_latents=False,
        start_layer=5,end_layer=8,layer_in=c,randomize_noise=False)[0]
    raw=checkpoint(generate,sf,ff,use_reentrant=False) if grad else generate(sf,ff)
    return raw_pre,raw,s,sf,ff

def reset_pretrained():
    sd=torch.load(b.opts.blending_checkpoint,map_location='cpu',weights_only=True)['model_state_dict']
    missing,unexpected=enc.load_state_dict(sd,strict=False)
    if unexpected or any(not k.startswith('clip_model.') for k in missing):
        raise RuntimeError('Pretrained Blending schema mismatch')
    pp.load_state_dict(torch.load(b.opts.pp_checkpoint,map_location='cpu',weights_only=True)['model_state_dict'],strict=True)
    enc.eval();pp.eval()



## ล็อก checkpoint epoch30 ทั้งสองโมเดลก่อนประเมิน
ไม่เลือกด้วยคะแนน validation หรือ test รอบนี้ และไม่ใช้ best.pt ทดแทน

In [ ]:
LOCK={};CK_PATHS={}
for family,root in [('HairFastGAN',STUDY_ROOT),('Barbershop-FT',FT_ROOT)]:
 CK_PATHS[family]={};LOCK[family]={}
 for t in 'ABCD':
  folder=root/('trial_'+t)
  candidates=[folder/'final_epoch30.pt',folder/'last.pt']
  path=next((p for p in candidates if p.is_file()),None)
  if path is None:raise FileNotFoundError(f'{family}/{t}: ไม่มี epoch30 หรือ last.pt; ห้ามแทนด้วย best.pt')
  ck=torch.load(path,map_location='cpu',weights_only=True)
  assert int(ck['epoch'])==30,f'{family}/{t}: ไม่ใช่ epoch30'
  if family=='HairFastGAN':
   assert ck['format']=='hairfast_bpp_pretrained_start_trial_v1'
   assert ck['config']['trial']==t and ck['config']['validation_keys']==VAL_KEYS
   assert ck['source_manifest_sha256']==sha(SOURCE/'manifest.json')
   assert ck['repo_commit']==REPO_COMMIT
   assert ck['start_blending_sha256']==REFERENCE_META['init_blending_sha256']
   assert 'blending_trainable_state_dict' in ck and 'post_process_state_dict' in ck
  else:
   assert ck['format']=='barbershop_shared_synthesis_ft_v2' and ck['protocol_id']==FT_ID
   assert ck['trial']==next(s for s in FT_TRIALS if s['name']==t)
   assert ck['pretrained_sha256']==FT_PROTOCOL['ffhq_sha256']
  CK_PATHS[family][t]=path;LOCK[family][t]=dict(path=str(path),sha256=sha(path),epoch=30)
  del ck
(OUT/'checkpoint_lock.json').write_text(json.dumps(LOCK,indent=2))
audit=[dict(item='checkpoint policy',HairFastGAN='fixed epoch30',Barbershop_FT='fixed epoch30',same=True),
 dict(item='historical detail gate',HairFastGAN=str(REFERENCE_META['config']['gates']),Barbershop_FT=str(FT_PROTOCOL['gates']),same=False),
 dict(item='historical losses',HairFastGAN=str(REFERENCE_META['config']['weights']),Barbershop_FT=str(FT_PROTOCOL['weights']),same=False),
 dict(item='train / validation cases',HairFastGAN='150 / 9',Barbershop_FT='150 / 9',same=True),
 dict(item='training modules',HairFastGAN='Blending / PostProcess',Barbershop_FT=str(FT_PROTOCOL['trainable']),same=False)]
pd.DataFrame(audit).to_csv(OUT/'protocol_audit.csv',index=False)


## เตรียม test ด้วยคู่เดิมและตรวจไม่ซ้ำกับ train/validation
ใช้ latent Barbershop เดิม ส่วน cache ของ HairFastGAN สร้างด้วย preprocessing เดิมและตรวจ crop ตรงกัน รายงานยังใช้ชื่อ test_candidate เพราะประวัติข้อมูลพัฒนาเก่าตรวจไม่ครบ

In [ ]:
if RUN_TEST:
    # TEST: no optimizer, no backward, no checkpoint selection from test scores.
    TEST_OUT=OUT/'test_candidate';TEST_OUT.mkdir(exist_ok=True)
    
    from pathlib import PurePosixPath
    if not DATASET_ZIP:
        print('Upload HairFastGAN_Dataset_Split_v2.zip containing the test images (not the review ZIP)')
        up=files.upload()
        if len(up)!=1:raise ValueError('Select one dataset ZIP')
        DATASET_ZIP=str(Path(next(iter(up))).resolve())
    DATASET_ZIP=Path(DATASET_ZIP)
    assert sha(DATASET_ZIP)=='5eb3a720c71989e6e36e1a38fae13840ec9e9853b5196893e190db499aa5719a','Dataset ZIP differs from reviewed v2'
    DATA_DIR=Path('/content')/('test_data_'+sha(DATASET_ZIP)[:12]);DATA_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as z:
        if sum(i.file_size for i in z.infolist())>600*1024**2:raise ValueError('Unexpected dataset size')
        for info in z.infolist():
            p=PurePosixPath(info.filename)
            if p.is_absolute() or '..' in p.parts or ((info.external_attr>>16)&0o170000)==0o120000:raise ValueError('Unsafe ZIP entry')
        z.extractall(DATA_DIR)
    mp=list(DATA_DIR.rglob('manifest.json'))
    assert len(mp)==1
    DATA_ROOT=mp[0].parent
    DATA_META=json.loads(mp[0].read_text());ITEMS=DATA_META['items'];ITEM_BY_ID={r['id']:r for r in ITEMS}
    for item in ITEMS:
        p=DATA_ROOT/item['file']
        assert p.resolve().is_relative_to(DATA_ROOT.resolve())
        assert sha(p)==item['sha256'],item['id']
    test_items=json.loads((DATA_ROOT/'test_candidate.json').read_text())
    assert len(test_items)==20
    for item in test_items:assert ITEM_BY_ID[item['id']]['split']=='test_candidate'
    trainval_cases=json.loads((SOURCE/'cases.json').read_text())
    def resolve_dataset_id(value,role):
        category={'face':'faces','shape':'styles','color':'colors'}[role]
        token=str(value);stem=Path(token).stem
        found=[r['id'] for r in ITEMS if r['category']==category and (r['id']==token or r['id']==stem or r['file']==token or r.get('original_path')==token)]
        if len(found)!=1:raise ValueError(f'Cannot resolve historical {role}: {value}')
        return found[0]
    used_ids={resolve_dataset_id(c[role],role) for c in trainval_cases if c['split'] in ['train','validation'] for role in ['face','shape','color']}
    used_ids.update(r['id'] for r in ITEMS if r['split'] in ['train','validation'])
    # Check across roles: same image bytes, decoded pixels, and declared identity/template groups.
    used=[ITEM_BY_ID[k] for k in used_ids]
    used_hashes={r['sha256'] for r in used};used_pixels={r.get('pixels') for r in used if r.get('pixels')}
    used_groups={r['group_id'] for r in used}
    for row in test_items:
        r=ITEM_BY_ID[row['id']]
        assert r['id'] not in used_ids and r['sha256'] not in used_hashes,('overlap',r['id'])
        assert r.get('pixels') not in used_pixels,('pixel overlap',r['id'])
        assert r['group_id'] not in used_groups,('group overlap',r['id'])
    faces=sorted([r['id'] for r in test_items if r['category']=='faces'])
    styles=sorted([r['id'] for r in test_items if r['category']=='styles'])
    colors=sorted([r['id'] for r in test_items if r['category']=='colors'])
    assert (len(faces),len(styles),len(colors))==(10,5,5)
    rng=random.Random(3407);rng.shuffle(styles);rng.shuffle(colors)
    TEST_CASES=[dict(key=f'test_{i:03d}',face=f,shape=styles[i%5],color=colors[i%5],face_group=ITEM_BY_ID[f]['group_id']) for i,f in enumerate(faces)]
    (TEST_OUT/'test_pairs.json').write_text(json.dumps(TEST_CASES,indent=2))
    TEST_KEYS=[r['key'] for r in TEST_CASES]
    
    assert [{k:r[k] for k in ['key','face','shape','color']} for r in TEST_CASES]==BS_CASES['test_candidate'],'Test pairs differ between models'


In [ ]:
if RUN_TEST:
    # Frozen preprocessing. Reuse official SEAN decode/e4e; do not use the custom trainable decoder.
    from utils.shape_predictor import align_face
    from models.encoder4editing.utils.model_utils import get_latents
    from models.sean_codes.models.pix2pix_model import encode_sean,decode_sean
    from models.Alignment import get_hair_face_code,get_new_shape
    TEST_CACHE=OUT/'test_cache';TEST_CACHE.mkdir()
    e4e=hair_fast.embed.e4e
    def erode_test(m,k=7):return -F.max_pool2d(-m.float(),k,1,k//2)
    def dilate_test(m,k=11):return F.max_pool2d(m.float(),k,1,k//2)
    @torch.no_grad()
    def test_embedding(ident):
        path=TEST_CACHE/(ident+'.pt')
        if path.exists():return gpu(torch.load(path,map_location='cpu',weights_only=True))
        with Image.open(DATA_ROOT/ITEM_BY_ID[ident]['file']) as im:original=im.convert('RGB').copy()
        image=align_face([original])[0].unsqueeze(0).cuda();emb=hair_fast.embed
        i512=emb.downsample_512(image);i256=emb.downsample_256(image)
        W=get_latents(e4e,emb.normalize(i256))
        output=emb.encoder.test(img=emb.normalize(image),return_latent=True)
        feature16=output[-1];S=output[-2]
        feature32=G([S],input_is_latent=True,return_latents=False,start_layer=3,end_layer=3,layer_in=feature16,randomize_noise=False)[0]
        mask=segment(emb.to_bisenet(i512))
        h=F.interpolate((mask==13).float(),(32,32),mode='bicubic')
        fw=G([W],input_is_latent=True,return_latents=False,start_layer=0,end_layer=3,randomize_noise=False)[0]
        feature32=feature32+.95*h*(fw-feature32)
        result=cpu(dict(W=W,F=feature32,S=S,mask=mask,image_256=i256,image_norm_256=emb.normalize(i256)))
        torch.save(result,path);return gpu(result)
    
    @torch.no_grad()
    def prepare_test_case(row):
        key=row['key']
        seed=(3407+int(hashlib.sha256(key.encode()).hexdigest()[:8],16))%(2**31)
        torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
        ef,es,ec=[test_embedding(row[r]) for r in ['face','shape','color']]
        face_code,_=get_hair_face_code(align.mask_generator,ef['mask'][0,0])
        _,hair_code=get_hair_face_code(align.mask_generator,es['mask'][0,0])
        labels=get_new_shape(align.mask_generator,face_code,hair_code)[None,None].long()
        target=labels==13;hf=(ef['mask']==13).float();hs=(es['mask']==13).float()
        codes=encode_sean(align.sean_model,torch.cat([ef['image_256'],es['image_256']],0),torch.cat([ef['mask'],es['mask']],0))
        gen1=decode_sean(align.sean_model,codes[0:1],labels)
        gen2=decode_sean(align.sean_model,codes[1:2],labels)
        aligned_embed=align.latent_encoder([gen1,gen2])
        a=aligned_embed['F'][0:1];c=aligned_embed['F'][1:2]
        masks=torch.cat([1-(1-hf)*(1-target.float()),target.float(),hs*target.float()],0)
        dil,ero=align.dilate_erosion.mask(masks)
        low=1-F.interpolate(torch.stack([dil[0],ero[1],ero[2]],0).float(),(32,32),mode='bicubic')
        feature=a+low[0]*(ef['F']-a);feature=c+low[1]*(feature-c);feature=es['F']+low[2]*(feature-es['F'])
        color_align=align.shape_module('face','color',{'face':ef,'color':ec})
        de=b.dilate_erosion.hair_from_mask(torch.cat([ef['mask'],ec['mask']],0))
        h1d=de[0][0].unsqueeze(0);h3d=de[0][1].unsqueeze(0);h3e=de[1][1].unsqueeze(0)
        hxd,_=b.dilate_erosion.mask(color_align['HM_X'])
        preserve=(1-h1d)*(1-h3d)*(1-hxd)
        protected=dilate_test(hf.bool()|target)<.5
        core=(erode_test(target)>.5)&~protected;refmask=erode_test(ec['mask']==13)>.5
        if min(int(core.sum()),int(refmask.sum()),int(protected.sum()))<100:raise ValueError('Insufficient mask pixels: '+key)
        aligned=G([ef['S']],input_is_latent=True,return_latents=False,start_layer=4,end_layer=8,layer_in=feature,randomize_noise=False)[0]
        d=dict(S=ef['S'],F=feature,I1=ef['image_norm_256'],face=ef['image_256'],shape=es['image_256'],ref=ec['image_256'],aligned=unit(aligned),target=target,protected=protected,core=core,refmask=refmask,eval_core=core,eval_refmask=refmask,
            args=[ef['S'][:,6:],ec['S'][:,6:],ef['image_norm_256']*preserve,ec['image_norm_256']*h3e])
        torch.save(cpu(d),TEST_CACHE/(key+'.pt'))
    


In [ ]:
if RUN_TEST:
 reset_pretrained()
 for row in TEST_CASES:
  prepare_test_case(row);print('Prepared shared test inputs:',row['key'])
SPLITS={'validation':VAL_KEYS}
if RUN_TEST:SPLITS['test_candidate']=TEST_KEYS

def common_case(key,split):
 if split=='validation':return gpu(load_case(key))
 assert split=='test_candidate' and RUN_TEST and key in TEST_KEYS
 return gpu(torch.load(TEST_CACHE/(key+'.pt'),map_location='cpu',weights_only=True))

# Compare original shared crops to the exact metric inputs. Use original native downsampling and clamp.
crop_rows=[]
for split,keys in SPLITS.items():
 assert [r['key'] for r in BS_CASES[split]]==keys
 for row in BS_CASES[split]:
  d=common_case(row['key'],split)
  for role,field in [('face','face'),('color','ref')]+([('shape','shape')] if split=='test_candidate' else []):
   with Image.open(CROPS/(row[role]+'.png')) as im:
    arr=np.asarray(im.convert('RGB')).copy()/255.
   x=torch.from_numpy(arr).float().permute(2,0,1)[None].cuda()
   small=hair_fast.embed.downsample_256(x).clamp(0,1)
   delta=float((small-d[field].clamp(0,1)).abs().max())
   assert delta<=1e-3,f'Crop mismatch {split}/{row["key"]}/{role}: {delta}'
   crop_rows.append(dict(split=split,case=row['key'],role=role,max_abs_delta=delta))
  del d,x,small
pd.DataFrame(crop_rows).to_csv(OUT/'crop_checks.csv',index=False)


## ฟังก์ชันเมทริกกลาง: เรียกฟังก์ชันเดียวสำหรับทุกภาพของทั้งสองโมเดล

In [ ]:
@torch.no_grad()
def common_metrics(full,d):
 im=to_size(full.clamp(0,1),256)
 d=dict(d,face=d['face'].clamp(0,1),ref=d['ref'].clamp(0,1))
 L,ab=quantile_color(im,d['ref'],d['eval_core'],d['eval_refmask'])
 identity=(id_features(im)*id_features(d['face'])).sum(1).mean()
 values=dict(L_quantile=float(L),ab_quantile=float(ab),identity_cosine=float(identity),
             protected_RGB_MAE_vs_input=float(masked_mae(im,d['face'],d['protected'])))
 values.update(extra_metrics(im,d))
 shape=shape_metrics(im,d)
 values.update({'proxy_'+k:v for k,v in shape.items()})
 assert all(np.isfinite(v) for v in values.values())
 return values

ROWS=[]
def record(family,trial,key,split,full):
 d=common_case(key,split);vals=common_metrics(full,d)
 ROWS.append(dict(family=family,trial=trial,epoch=0 if trial=='Baseline' else 30,case=key,split=split,**vals))
 folder=OUT/split/'images'/key;folder.mkdir(parents=True,exist_ok=True)
 saveim(full,folder/(family+'_'+trial+'.png'))
 if family=='HairFastGAN' and trial=='Baseline':
  for field in ['face','ref']:saveim(d[field],folder/(field+'.png'))
  row=next(r for r in BS_CASES[split] if r['key']==key)
  shutil.copy2(CROPS/(row['shape']+'.png'),folder/'shape.png')
  for field in ['target','protected','eval_core','eval_refmask']:saveim(d[field].float().repeat(1,3,1,1),folder/(field+'.png'))
 pd.DataFrame(ROWS).to_csv(OUT/'per_case_metrics.partial.csv',index=False)
 print(family,trial,split,key,flush=True)

torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
with torch.no_grad():
 for trial in ['Baseline','A','B','C','D']:
  reset_pretrained()
  if trial!='Baseline':
   path=CK_PATHS['HairFastGAN'][trial];assert sha(path)==LOCK['HairFastGAN'][trial]['sha256']
   ck=torch.load(path,map_location='cpu',weights_only=True)
   sd=ck['blending_trainable_state_dict']
   assert set(sd)=={n for n,p in enc.named_parameters() if n.startswith('modulation_module_list.')}
   state=enc.state_dict();state.update(sd);enc.load_state_dict(state,strict=True)
   pp.load_state_dict(ck['post_process_state_dict'],strict=True);del ck,sd,state
  for split,keys in SPLITS.items():
   for key in keys:
    d=common_case(key,split);pre,raw,s,sf,ff=joint_forward(d)
    record('HairFastGAN',trial,key,split,((raw+1)/2).clamp(0,1))
    del d,pre,raw,s,sf,ff
reset_pretrained()
for module in [enc,pp,G,align,hair_fast.embed.encoder,hair_fast.embed.e4e]:module.cpu()
id_net.cuda().eval();parser_net.cuda().eval();gc.collect();torch.cuda.empty_cache()


## ประเมิน Barbershop-FT ผ่านตัวประเมินกลางชุดเดียวกัน

In [ ]:
BARBER_REPO=Path('/content/Barbershop_common_eval');BARBER_REV=FT_PROTOCOL['barbershop_revision']
if not (BARBER_REPO/'.git').is_dir():subprocess.run(['git','clone','https://github.com/ZPdesu/Barbershop.git',str(BARBER_REPO)],check=True)
subprocess.run(['git','-C',str(BARBER_REPO),'checkout',BARBER_REV],check=True)
for kind in ['fused_act','upfirdn2d']:
 relative='models/stylegan2/op/'+kind+'.py'
 original=subprocess.check_output(['git','-C',str(BARBER_REPO),'show',BARBER_REV+':'+relative],text=True)
 (BARBER_REPO/relative).write_text(_native_stylegan_source(original,kind))


In [ ]:
import importlib
private=Path('/content/bs_ft_private');package=private/'bs_ft_synthesis';package.mkdir(parents=True,exist_ok=True)
private=Path('/content/bs_ft_private');package=private/'bs_ft_synthesis';package.mkdir(parents=True,exist_ok=True)
private=Path('/content/bs_ft_private');package=private/'bs_ft_synthesis';package.mkdir(parents=True,exist_ok=True)
(package/'__init__.py').write_text('')
shutil.copytree(BARBER_REPO/'models/stylegan2/op',package/'op',dirs_exist_ok=True)
model_source=(BARBER_REPO/'models/stylegan2/model.py').read_text().replace('from models.stylegan2.op import','from .op import')
(package/'model.py').write_text(model_source)
if str(private) not in sys.path:sys.path.insert(0,str(private))
BSGenerator=importlib.import_module('bs_ft_synthesis.model').Generator
bs=BSGenerator(1024,512,8,channel_multiplier=2)
init=torch.load(BARBER_WEIGHTS/'ffhq.pt',map_location='cpu',weights_only=False)['g_ema']
bs.load_state_dict(init,strict=True);del init
bs.load_state_dict(init,strict=True);del init
bs.eval().requires_grad_(False)
for module in list(bs.convs)[6:]+list(bs.to_rgbs)[3:]:module.requires_grad_(True)
NAMED=dict(bs.named_parameters());NAMES=[n for n,p in NAMED.items() if p.requires_grad]
NAMED=dict(bs.named_parameters());NAMES=[n for n,p in NAMED.items() if p.requires_grad]
CONV_NAMES=[n for n in NAMES if n.startswith('convs.')];RGB_NAMES=[n for n in NAMES if n.startswith('to_rgbs.')]
CONV_NAMES=[n for n in NAMES if n.startswith('convs.')];RGB_NAMES=[n for n in NAMES if n.startswith('to_rgbs.')]
assert CONV_NAMES and RGB_NAMES and set(NAMES)==set(CONV_NAMES+RGB_NAMES)
INITIAL={n:NAMED[n].detach().cpu().clone() for n in NAMES}
def state_hash(values):
    h=hashlib.sha256()
    for n,v in sorted(values.items()):h.update(n.encode());h.update(v.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()
def frozen_hash():return state_hash({n:v for n,v in bs.state_dict().items() if n not in NAMES})
FROZEN_HASH=frozen_hash();INITIAL_HASH=state_hash(INITIAL)
FROZEN_HASH=frozen_hash();INITIAL_HASH=state_hash(INITIAL)
bs.cuda();id_net.cuda().eval().requires_grad_(False);parser_net.cuda().eval().requires_grad_(False)
bs.cuda();id_net.cuda().eval().requires_grad_(False);parser_net.cuda().eval().requires_grad_(False)
bs.cuda();id_net.cuda().eval().requires_grad_(False);parser_net.cuda().eval().requires_grad_(False)
torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
torch.backends.cudnn.benchmark=False;torch.backends.cudnn.deterministic=True
torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
torch.backends.cuda.matmul.allow_tf32=False;torch.backends.cudnn.allow_tf32=False
def load_trainable(values):
    assert set(values)==set(NAMES)
    with torch.no_grad():
        for n in NAMES:NAMED[n].copy_(values[n].to(NAMED[n]))
def trainable_state():return {n:NAMED[n].detach().cpu().clone() for n in NAMES}
def bs_render(S,feat,grad=False):
    # Equivalent to official generator start_layer=4/end_layer=8; checkpoint each resolution.
    out=feat;skip=None
    for j in range(3,len(bs.to_rgbs)):
        i=1+2*j
        def block(x,w,rgb,j=j,i=i):
            x=bs.convs[2*j](x,w[:,i],noise=getattr(bs.noises,f'noise_{1+2*j}'))
            x=bs.convs[2*j+1](x,w[:,i+1],noise=getattr(bs.noises,f'noise_{2+2*j}'))
            return x,bs.to_rgbs[j](x,w[:,i+2],rgb)
        out,skip=checkpoint(block,out,S,skip,use_reentrant=False) if grad else block(out,S,skip)
    return skip
def latent(key,split):
    rec=front_record(key,split);folder=Path(rec['final_png']).parent/'Blend_realistic'
    paths=list(folder.glob('*.npz'));assert len(paths)==1
    with np.load(paths[0]) as z:S=torch.from_numpy(z['latent_in'].copy()).cuda();feat=torch.from_numpy(z['latent_F'].copy()).cuda()
    return S.detach(),feat.detach()

def front_record(key,split):return FRONT_INDEX[split][key]
bs.eval().requires_grad_(False)
with torch.no_grad():
 for trial in ['Baseline','A','B','C','D']:
  load_trainable(INITIAL)
  if trial!='Baseline':
   path=CK_PATHS['Barbershop-FT'][trial];assert sha(path)==LOCK['Barbershop-FT'][trial]['sha256']
   ck=torch.load(path,map_location='cpu',weights_only=True)
   assert ck['initial_sha256']==INITIAL_HASH
   load_trainable(ck['state_dict']);del ck
  before=state_hash(trainable_state())
  for split,keys in SPLITS.items():
   for key in keys:
    S,feat=latent(key,split);raw=bs_render(S,feat,grad=False)
    if trial=='Baseline':
     cached=torch.from_numpy(np.load(FRONT_INDEX[split][key]['final_float'])).to(raw)
     torch.testing.assert_close(((raw+1)/2).clamp(0,1),cached,atol=2e-4,rtol=2e-4)
     del cached
    record('Barbershop-FT',trial,key,split,((raw+1)/2).clamp(0,1));del S,feat,raw
  assert state_hash(trainable_state())==before and frozen_hash()==FROZEN_HASH
load_trainable(INITIAL)
for family,entries in LOCK.items():
 for trial,r in entries.items():assert sha(Path(r['path']))==r['sha256']
frame=pd.DataFrame(ROWS)
assert len(frame)==10*sum(map(len,SPLITS.values())) and not frame.duplicated(['family','trial','split','case']).any()
frame.to_csv(OUT/'per_case_metrics.csv',index=False)
(OUT/'per_case_metrics.partial.csv').unlink()


## กราฟรูปแบบเดียวกัน ตารางเปรียบเทียบ และภาพคู่เดียวกัน
แยก validation/test และแยกเมทริกหลักออกจาก HFG-derived shape proxy ไม่ใช้คะแนน test เลือก checkpoint

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
MAIN=[('L_quantile','Lightness error L ↓'),('ab_quantile','Color error ab ↓'),
 ('identity_cosine','Face identity cosine ↑'),('protected_RGB_MAE_vs_input','Protected-region MAE ↓'),
 ('protected_SSIM','Protected-region SSIM ↑'),('protected_PSNR','Protected-region PSNR ↑')]
PROXY=[('proxy_target_mask_IoU','Proxy hair-mask IoU ↑'),('proxy_target_Dice','Proxy hair-mask Dice ↑'),
 ('proxy_boundary_F1_2px','Proxy hair boundary F1 ↑')]
FAMILIES=['HairFastGAN','Barbershop-FT'];TRIALS=['Baseline','A','B','C','D']
PALETTE={'HairFastGAN':'#2875BA','Barbershop-FT':'#D77A29'}
plt.rcParams.update({'font.family':'DejaVu Sans','font.size':10,'figure.facecolor':'white'})
metrics=[k for k,_ in MAIN+PROXY]
means=frame.groupby(['split','family','trial'])[metrics].mean()
sds=frame.groupby(['split','family','trial'])[metrics].std(ddof=1)
means.to_csv(OUT/'mean_metrics.csv');sds.to_csv(OUT/'sd_across_cases.csv')
paired=[]
for split,keys in SPLITS.items():
 for trial in TRIALS:
  a=frame[(frame.split==split)&(frame.family==FAMILIES[0])&(frame.trial==trial)].set_index('case')
  b_=frame[(frame.split==split)&(frame.family==FAMILIES[1])&(frame.trial==trial)].set_index('case').reindex(a.index)
  assert len(a)==len(keys) and not b_[metrics].isna().any().any()
  for key in a.index:paired.append(dict(split=split,trial=trial,case=key,**{m:float(b_.loc[key,m]-a.loc[key,m]) for m in metrics}))
pd.DataFrame(paired).to_csv(OUT/'paired_deltas_Barbershop_minus_HairFastGAN.csv',index=False)
# Delta per case; arrows in charts determine whether a positive delta is favorable.
with PdfPages(OUT/'common_comparison_report.pdf') as pdf:
 for split,keys in SPLITS.items():
  folder=OUT/split;folder.mkdir(exist_ok=True)
  for group,specs in [('primary',MAIN),('shape_proxy',PROXY)]:
   # Same limits for separate and combined figures within this split/metric.
   limits={}
   for metric,_ in specs:
    low=min(float(means.loc[(split,f,t),metric]-sds.loc[(split,f,t),metric]) for f in FAMILIES for t in TRIALS)
    high=max(float(means.loc[(split,f,t),metric]+sds.loc[(split,f,t),metric]) for f in FAMILIES for t in TRIALS)
    margin=max((high-low)*.12,1e-4);limits[metric]=(max(0,low-margin),high+margin)
   for families,label in [([FAMILIES[0]],'HairFastGAN'),([FAMILIES[1]],'Barbershop-FT'),(FAMILIES,'combined')]:
    fig,axs=plt.subplots(len(specs)//3,3,figsize=(15,4.3*(len(specs)//3)),squeeze=False)
    for ax,(metric,title) in zip(axs.flat,specs):
     for j,family in enumerate(families):
      offset=0 if len(families)==1 else (-.12 if j==0 else .12)
      x=np.arange(5)+offset
      y=[means.loc[(split,family,t),metric] for t in TRIALS]
      err=[sds.loc[(split,family,t),metric] for t in TRIALS]
      ax.errorbar(x,y,yerr=err,fmt='o',capsize=3,color=PALETTE[family],label=family,markersize=6)
     ax.set_xticks(range(5),['Baseline\nep 0']+[t+'\nep 30' for t in 'ABCD'])
     ax.set_ylim(*limits[metric]);ax.set_title(title,loc='left');ax.grid(alpha=.2)
     ax.spines[['top','right']].set_visible(False)
    axs[0,0].legend(fontsize=8)
    suffix=' | HFG-derived targets; not independent ground truth' if group=='shape_proxy' else ''
    fig.suptitle(f'{label} | {split} | Mean ± SD across {len(keys)} cases'+suffix,fontsize=13)
    fig.tight_layout(rect=(0,0,1,.93));fig.savefig(folder/(label+'_'+group+'.png'),dpi=180)
    pdf.savefig(fig);plt.show();plt.close(fig)
  for key in keys:
   image_dir=folder/'images'/key
   fig,axs=plt.subplots(3,5,figsize=(17,11))
   for ax in axs.flat:ax.axis('off')
   for j,(name,label) in enumerate([('face','Input face'),('shape','Shape reference'),('ref','Color reference')]):
    with Image.open(image_dir/(name+'.png')) as im:axs[0,j].imshow(im)
    axs[0,j].set_title(label)
   for i,family in enumerate(FAMILIES,1):
    for j,trial in enumerate(TRIALS):
     with Image.open(image_dir/(family+'_'+trial+'.png')) as im:axs[i,j].imshow(im)
     axs[i,j].set_title(f'{family}\n{trial} | epoch {0 if trial=="Baseline" else 30}',fontsize=10)
   fig.suptitle(f'{split} / {key} | Shared inputs and evaluation; fixed epochs')
   fig.tight_layout();fig.savefig(folder/(key+'_comparison.png'),dpi=150);pdf.savefig(fig);plt.close(fig)

protocol=dict(evaluation='retrospective fixed-epoch comparison; not common best-checkpoint reselection',
 checkpoint_rule='baseline epoch0 and trials A-D epoch30 for BOTH models; no ranking or selection from test',
 common_metrics=MAIN,shape_proxy_metrics=PROXY,
 excluded_from_cross_model_ranking=['training loss','detail vs model-specific reference','model-specific normalized score'],
 splits=SPLITS,test_status='previously viewed candidate test; not new untouched confirmatory test',
 pixel_policy='float generator RGB clamped [0,1], bilinear 256px, same cached references and masks; PNG only for display',
 limitations=['historical training losses and modules differ','historical checkpoint gates differ',
 'shape targets derived from HairFastGAN','150 pairs / 9 validation cases; test is 10 faces with repeated styles/colors',
 'equal epochs/updates do not imply equal compute','no independent ground-truth output or human quality annotation'],
 environment=dict(python=sys.version,torch=torch.__version__,cuda=torch.version.cuda,gpu=torch.cuda.get_device_name(0),hairfast_commit=REPO_COMMIT,barbershop_commit=BARBER_REV))
(OUT/'common_protocol.json').write_text(json.dumps(protocol,indent=2,ensure_ascii=False))
(OUT/'README_TH.txt').write_text('''รายงานเปรียบเทียบเพิ่มเติมหลังเห็นผลเดิม
ทั้งสองโมเดลใช้ baseline และ A-D epoch30 พร้อมฟังก์ชันเมทริกเดียวกัน
ดู protocol_audit.csv เพื่อรายงานความต่างของการฝึกเดิมอย่างตรงไปตรงมา
ไม่อ้างว่าเป็นการทดลองที่ฝึกเหมือนกันทุกอย่าง หรือ best checkpoint ที่เลือกด้วยกติกาเดียวกัน
shape proxy ไม่ใช่ ground truth อิสระ ไม่ใช้ detail คนละฐานหรือ training loss ตัดสินข้ามโมเดล
กราฟแสดง SD ระหว่างกรณี ไม่ใช่ confidence interval หรือผลทดสอบนัยสำคัญ
validation กับ test แยกไฟล์/โฟลเดอร์; ไม่เลือกหรือปรับพารามิเตอร์จาก test
หากต้องการงานทดลองใหม่ที่มีเกณฑ์เลือก best เดียวกัน ต้องกำหนดเกณฑ์ก่อนฝึกและบันทึก checkpoint ทุก epoch
''',encoding='utf-8')
(OUT/'completion.json').write_text(json.dumps(dict(status='complete',cases_by_split={k:len(v) for k,v in SPLITS.items()},metric_rows=len(frame),no_training=True,no_reselection=True,checkpoints_unchanged=True),indent=2))
display(means)
# Exclude large reproducibility cache from downloadable report; cache remains in Drive.
archive=Path(str(OUT)+'.zip')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
 for path in sorted(OUT.rglob('*')):
  if path.is_file() and 'test_cache' not in path.relative_to(OUT).parts:z.write(path,path.relative_to(OUT))
print('Saved report:',archive)
files.download(str(archive))
